# KTTS single-speaker adaptation of Bolbosh Matcha-TTS
Fine-tunes the MIT-licensed GAASH Lab Kashmiri checkpoint on the CC BY-NC 3.0 KTTS corpus. Outputs are non-commercial and require attribution.

In [ ]:
!nvidia-smi
!git clone -q --depth 1 https://github.com/sameerahmad86232/cipher.git /kaggle/working/cipher
!wget -q https://github.com/gaash-lab/Bolbosh/archive/refs/heads/main.zip -O /kaggle/working/bolbosh.zip
!unzip -q /kaggle/working/bolbosh.zip -d /kaggle/working
%cd /kaggle/working/Bolbosh-main
!pip install -q 'numpy<2' Cython soundfile scipy pyarrow huggingface_hub lightning hydra-core hydra-colorlog rootutils rich torchmetrics tensorboard librosa einops inflect Unidecode conformer diffusers peft jiwer
!pip install -q git+https://github.com/abdulmuizz0903/KashmiriNormalizer
!python setup.py build_ext --inplace -q

In [ ]:
!python /kaggle/working/cipher/scripts/prepare-ktts-matcha.py --repo-dir /kaggle/working/Bolbosh-main --output /kaggle/working/ktts-matcha
from pathlib import Path
import json
report = json.loads(Path('/kaggle/working/ktts-matcha/preparation-report.json').read_text())
assert report['accepted'] >= 2500 and report['hours'] >= 3.5, report
print(report)

In [ ]:
from huggingface_hub import hf_hub_download
import torch
base = hf_hub_download('GAASH-Lab/Matcha-TTS-Kashmiri', 'model.ckpt')
checkpoint = torch.load(base, map_location='cpu', weights_only=False)
checkpoint['epoch'] = -1
checkpoint['global_step'] = 0
checkpoint['optimizer_states'] = []
checkpoint['lr_schedulers'] = []
checkpoint.pop('loops', None)
warmstart = '/kaggle/working/bolbosh-ktts-warmstart.ckpt'
torch.save(checkpoint, warmstart)
print('Prepared weights-only warm start:', warmstart)

In [ ]:
%cd /kaggle/working/Bolbosh-main
!python matcha/train.py data=kashmiri model=matcha trainer=gpu logger=tensorboard ckpt_path=/kaggle/working/bolbosh-ktts-warmstart.ckpt train=true test=false run_name=ktts_voice_v1 data.batch_size=16 data.num_workers=2 trainer.max_epochs=30 trainer.precision=16-mixed +trainer.accumulate_grad_batches=2 paths.output_dir=/kaggle/working/bolbosh-ktts-output

In [ ]:
from pathlib import Path
import shutil
root = Path('/kaggle/working/bolbosh-ktts-output')
checkpoints = sorted(root.rglob('*.ckpt'), key=lambda p: p.stat().st_mtime)
assert checkpoints, 'Training produced no checkpoint'
release = Path('/kaggle/working/kashmiri-bolbosh-ktts-v1')
release.mkdir(exist_ok=True)
shutil.copy2(checkpoints[-1], release / 'model.ckpt')
shutil.copy2('/kaggle/working/ktts-matcha/preparation-report.json', release / 'preparation-report.json')
(release / 'LICENSE-DATA.txt').write_text('KTTS data: CC BY-NC 3.0. Non-commercial use only. DOI: 10.17632/5c4dcvxdmb.2\n')
shutil.make_archive('/kaggle/working/kashmiri-bolbosh-ktts-v1', 'zip', release)
print('Artifact:', '/kaggle/working/kashmiri-bolbosh-ktts-v1.zip')